# Build a Kokoro voice from your recording (KVoiceWalk)

Throwaway spike for the LekturaC Kokoro tester. Makes `dev.bin`, a Kokoro voicepack that sounds like the voice in your recording.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells in order (Shift+Enter). Cell 3 asks you to pick your WAV. Cell 5 is the long one.

Your recording is uploaded to this Colab session (Google's servers) and is deleted when the session ends.

## 1. Check the GPU

In [ ]:
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again.'

## 2. Install KVoiceWalk (about 3 to 5 minutes)
Kokoro needs Python 3.12 or older, and Colab's Python is newer, so this makes a separate Python 3.12 just for KVoiceWalk (with `uv`).

In [ ]:
import os
%cd /content
!apt-get -qq install -y espeak-ng > /dev/null
!rm -rf /content/kvoicewalk && git clone -q --depth 1 https://github.com/RobViren/kvoicewalk.git /content/kvoicewalk
assert os.path.exists('/content/kvoicewalk/main.py'), 'Download of KVoiceWalk failed. Run this cell again.'
!pip install -q uv librosa soundfile
%cd /content/kvoicewalk
!uv sync --python 3.12 --quiet
!.venv/bin/python -c "import torch, kokoro; print('KVoiceWalk Python ready | GPU:', torch.cuda.is_available())"
assert os.path.exists('/content/kvoicewalk/.venv/bin/python'), 'Install failed. Paste the red text above.'
%cd /content
print('Installed. If you had already uploaded your WAV, run step 3 again (this step clears it).')

## 3. Upload your recording
Pick the WAV you downloaded from the tester (any WAV works; it's converted to 24 kHz mono here).

In [ ]:
import os, librosa, soundfile as sf
from google.colab import files
os.makedirs('/content/kvoicewalk/in', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
audio, _ = librosa.load(name, sr=24000, mono=True)
sf.write('/content/kvoicewalk/in/dev.wav', audio, 24000)
seconds = len(audio) / 24000
print(f'{seconds:.1f} s, 24 kHz mono')
if seconds < 15:
    print('Short. 20-30 s gives a better voice; consider recording again.')

## 4. What you said

If you read the tester's passage **word for word**, leave this as it is.
If you said something else, set `USE_WHISPER = True` and it will be transcribed for you.

In [ ]:
TARGET_TEXT = "Good morning, everyone. Today we're going to look at how living things get their energy. Think about the last thing you ate. Where did that energy come from before it reached your plate? Every food chain starts with the sun. Plants catch its light, store it as sugar, and pass it on to the animals that eat them. By the end of this lesson, you'll be able to draw a simple food chain and explain each link. Ready? Let's begin."
USE_WHISPER = False

os.makedirs('/content/kvoicewalk/texts', exist_ok=True)
with open('/content/kvoicewalk/texts/dev.txt', 'w') as f:
    f.write(TARGET_TEXT)
print('Saved.')

## 5. Build the voice (the long step)

It prints in three parts:
1. **Stock voices** rated against your recording (a few minutes).
2. **Interpolation search**: a long list of voice pairs such as `0 1 -1.50 af_heart ...`. This is the slow part and **saves nothing yet**.
3. **The walk**: `Step:` lines each time it gets closer, and `Progress: step N/10000` every 100 steps. `best_target_sim` is how close it sounds to you (0 to 1; the KVoiceWalk author got 0.93 after 10,000 steps).

**You can stop it early** (the ■ button), but only once part 3 has printed at least one `Step:` line. Every improvement is saved from then on, so step 6 still works.
Free Colab disconnects after a while if the tab sits idle, so keep the tab open and look in now and then.

In [ ]:
STEPS = 10000
%cd /content/kvoicewalk
whisper = '--transcribe_start' if USE_WHISPER else ''
!.venv/bin/python main.py --target_text ./texts/dev.txt --target_audio ./in/dev.wav {whisper} --interpolate_start --step_limit {STEPS} --output_name dev --device cuda --log_interval 100

## 6. Listen to the best result and download `dev.bin`

In [ ]:
import glob, re, IPython.display as ipd

pts = glob.glob('/content/kvoicewalk/out/dev_dev_*/dev_*.pt')
assert pts, ('No voice saved yet. Step 5 saves its first file at its first "Step:" line, '
             'after the interpolation search (the long list of voice pairs) has finished. Let step 5 run longer.')
step = lambda p: int(re.match(r'dev_(\d+)_', os.path.basename(p)).group(1))
latest_run = max(os.path.dirname(p) for p in pts)
best = max((p for p in pts if os.path.dirname(p) == latest_run), key=step)
print('Best so far:', os.path.basename(best))
ipd.display(ipd.Audio(best[:-3] + '.wav'))

voice = torch.load(best, map_location='cpu', weights_only=True)
print('shape', tuple(voice.shape))
assert voice.numel() == 510 * 256, 'Unexpected voice shape'
voice.float().contiguous().numpy().astype('<f4').tofile('/content/dev.bin')
assert os.path.getsize('/content/dev.bin') == 522240
files.download('/content/dev.bin')

## 7. Use it in the tester

Put `dev.bin` at `spikes/kokoro-tester/voices/dev.bin` and reload the tester: a **Load developer voice** button appears in section 2.
Or open "Add a custom voice" in section 2 and pick the file.